This notebook loads all the parquet files into a pandas dataframes.

It also create a pkl file so you can use these dataframes in another notebook without making changes here

In [37]:
import pandas as pd

# Load Artist related data
artists = pd.read_parquet('./data/mb_artist.parquet')
artist_tags = pd.read_parquet('./data/mb_artist_tag.parquet') # Updated name
artist_ratings = pd.read_parquet('./data/mb_artist_ratings.parquet')

# Load Album related data
albums = pd.read_parquet('./data/mb_album.parquet')
#album_tags = pd.read_parquet('./data/mb_album_tag_map.parquet')
album_ratings = pd.read_parquet('./data/mb_album_ratings.parquet')

# Verify the loads
dataframes = {
    "Artists": artists, 
    "Artist Tags": artist_tags, 
    "Artist Ratings": artist_ratings,
    "Albums": albums, 
    #"Album Tags": album_tags, 
    "Album Ratings": album_ratings
}

for name, df in dataframes.items():
    print(f"✅ {name}: {df.shape[0]:,} rows loaded.")

✅ Artists: 2,867,969 rows loaded.
✅ Artist Tags: 731,552 rows loaded.
✅ Artist Ratings: 77,571 rows loaded.
✅ Albums: 2,241,402 rows loaded.
✅ Album Ratings: 143,733 rows loaded.


In [38]:
# Join Artists with Tags
artist_tag_joined = pd.merge(
    artists, 
    artist_tags, 
    left_on='id', 
    right_on='artist_id', 
    how='inner'
)

# Join with Ratings (Left join to keep artists even if unrated)
final_artist_df = pd.merge(
    artist_tag_joined,
    artist_ratings,
    on='artist_id',
    how='left'
).drop(columns=['artist_id']) # Clean up duplicate ID

# Fill missing ratings so math doesn't break
final_artist_df['rating'] = final_artist_df['rating'].fillna(0)
final_artist_df['rating_count'] = final_artist_df['rating_count'].fillna(0)

In [39]:
# Join Albums with Ratings
final_album_df = pd.merge(
    albums, 
    album_ratings,
    left_on='id', 
    right_on='album_id', 
    how='inner'
)



# Fill missing ratings
final_album_df['rating'] = final_album_df['rating'].fillna(0)
final_album_df['rating_count'] = final_album_df['rating_count'].fillna(0)

In [40]:
# Rename artist columns for clarity in the master table
artists_prep = final_artist_df.rename(columns={
    'name': 'artist_name',
    'gid': 'artist_gid',
    'area': 'artist_area',
    'tag_id': 'artist_tag_id',
    'tag_count': 'artist_tag_count',
    'rating': 'artist_rating',
    'rating_count': 'artist_rating_count'
})

# Final Join: Link Albums to their Artists
master_df = pd.merge(
    final_album_df,
    artists_prep,
    left_on='artist_credit', # The ID of the artist on the album record
    right_on='id',           # The ID in the artist table
    how='inner',
    suffixes=('_album', '_artist_meta')
).drop(columns=['id_artist_meta'])

# Final rename for album-specific columns
master_df = master_df.rename(columns={
    'id_album': 'album_id',
    'name': 'album_name',
    'tag_id': 'album_tag_id',
    'tag_count': 'album_tag_count',
    'rating': 'album_rating',
    'rating_count': 'album_rating_count'
})

print(f"\n🚀 Master DataFrame ready with {master_df.shape[0]:,} rows.")


🚀 Master DataFrame ready with 3,049,815 rows.


In [44]:
# --- مرحله ۱: آماده‌سازی و Match کردن آیدی‌ها (کد خودت با کمی بهینه‌سازی) ---
def normalize(s):
    if pd.isna(s): return ''
    return str(s).strip().lower()

ARTIST_NAME_COL = 'name'
final_artist_df['_name_norm'] = final_artist_df[ARTIST_NAME_COL].apply(normalize)
lastfm_artists['_name_norm'] = lastfm_artists['lastfm_name'].apply(normalize)

# وصل کردن آیدی‌ها
final_artist_df = final_artist_df.merge(
    lastfm_artists[['lastfm_id', '_name_norm']],
    on='_name_norm',
    how='left'
).drop(columns=['_name_norm'])

# مدیریت ستون‌های تکراری و تبدیل تایپ (کد خودت)
if 'lastfm_id_x' in final_artist_df.columns:
    final_artist_df['lastfm_id'] = final_artist_df['lastfm_id_x'].fillna(final_artist_df.get('lastfm_id_y', pd.NA))
    cols_to_drop = [c for c in ['lastfm_id_x', 'lastfm_id_y'] if c in final_artist_df.columns]
    final_artist_df = final_artist_df.drop(columns=cols_to_drop)

final_artist_df['lastfm_id'] = pd.array(final_artist_df['lastfm_id'], dtype=pd.Int64Dtype())

# --- مرحله ۲: اضافه کردن وزن‌ها (بخش جدید) ---
# حالا که lastfm_id داریم، می‌توانیم وزن‌ها را بیاوریم
weights_df = pd.read_csv('./data/user_artists.dat', sep='\t', usecols=['artistID', 'weight']).rename(columns={'artistID': 'lastfm_id'})
weights_grouped = weights_df.groupby('lastfm_id')['weight'].sum().reset_index()

# چسباندن وزن به جدول اصلی
final_artist_df = pd.merge(final_artist_df, weights_grouped, on='lastfm_id', how='left')
final_artist_df['weight'] = final_artist_df['weight'].fillna(0)

# حذف تکراری‌های احتمالی هنرمند (بسیار مهم)
final_artist_df = final_artist_df.drop_duplicates(subset=['id'], keep='first')

# --- مرحله ۳: گزارش نهایی ---
total = len(final_artist_df)
matched = final_artist_df['lastfm_id'].notna().sum()
print(f'✅ Total artists: {total:,}')
print(f'✅ Matched with Last.fm: {matched:,} ({matched/total*100:.1f}%)')
print(f'✅ Max Weight found: {final_artist_df["weight"].max():,}')

final_artist_df[[ARTIST_NAME_COL, 'lastfm_id', 'weight']].head(10)

✅ Total artists: 253,032
✅ Matched with Last.fm: 12,230 (4.8%)
✅ Max Weight found: 2,393,140.0


,name,lastfm_id,weight
0,Various Artists,15457,1581.0
328,Massive Attack,238,42195.0
354,Apartment 26,13180,480.0
363,Dr. Evil,<NA>,0.0
367,Robert Miles,2745,3723.0
381,Vincent Gallo,<NA>,0.0
383,Squirrel Nut Zippers,8066,328.0
387,Giant Sand,4412,122.0
392,Éric Serra,<NA>,0.0
396,William S. Burroughs,<NA>,0.0


In [46]:
LASTFM_PATH = './data/artists.dat'   
WEIGHT_PATH = './data/user_artists.dat'


lastfm_artists = pd.read_csv(
    LASTFM_PATH,
    sep='\t',
    usecols=['id', 'name'],
    dtype={'id': int, 'name': str}
).rename(columns={'id': 'lastfm_id', 'name': 'lastfm_name'})

weights_df = pd.read_csv(
    WEIGHT_PATH, 
    sep='\t', 
    usecols=['artistID', 'weight']
).rename(columns={'artistID': 'lastfm_id'})
weights_grouped = weights_df.groupby('lastfm_id')['weight'].sum().reset_index()

cols_to_remove = ['weight', 'weight_x', 'weight_y']
for col in cols_to_remove:
    if col in final_artist_df.columns:
        final_artist_df = final_artist_df.drop(columns=[col])


final_artist_df = pd.merge(
    final_artist_df, 
    weights_grouped, 
    on='lastfm_id', 
    how='left'
)

print(f'✅ Last.fm artists loaded: {len(lastfm_artists):,} rows')

lastfm_artists.head(3)
weights_grouped.head(3)

✅ Last.fm artists loaded: 17,632 rows


,lastfm_id,weight
0,1,771
1,2,8012
2,3,775


This notebook loads all the parquet files into a pandas dataframes.

It also create a pkl file so you can use these dataframes in another notebook without making changes here

In [ ]:
import os

# Create a folder for your pickles if it doesn't exist
os.makedirs('./data/pickles', exist_ok=True)

final_artist_df = final_artist_df.drop_duplicates(subset=['id'], keep='first')
# Save the 3 join tables
final_artist_df.to_pickle('./data/pickles/final_artist_df.pkl')
final_album_df.to_pickle('./data/pickles/final_album_df.pkl')
master_df.to_pickle('./data/pickles/master_df.pkl')

print("✅ Artist, Album, and Master tables have been pickled!")

✅ Artist, Album, and Master tables have been pickled!


In [48]:
print(f'\nfinal_artist_df columns: {final_artist_df.columns.tolist()}')


final_artist_df columns: ['id', 'name', 'begin_date_year', 'type', 'area', 'gender', 'tag_id', 'tag_count', 'rating', 'rating_count', 'lastfm_id', 'weight']


In [49]:
pkl = pd.read_pickle('./data/pickles/final_artist_df.pkl')
print(pkl.loc[pkl['name'] == 'Robert Miles', ['id', 'name', 'lastfm_id', 'weight']])

   id          name  lastfm_id  weight
4   9  Robert Miles       2745  3723.0


In [50]:
print(pkl)

             id                  name  begin_date_year  type     area  gender  \
0             1       Various Artists              NaN   3.0      NaN     NaN   
1             4        Massive Attack           1987.0   2.0    221.0     NaN   
2             6          Apartment 26           1999.0   2.0    221.0     NaN   
3             7              Dr. Evil              NaN   4.0      NaN     1.0   
4             9          Robert Miles           1969.0   1.0    105.0     1.0   
...         ...                   ...              ...   ...      ...     ...   
253027  3275005     A Familia Monstro              NaN   2.0  99488.0     NaN   
253028  3275026               Hufhout           2022.0   2.0    462.0     NaN   
253029  3275038  The James Range Band           2024.0   2.0    379.0     NaN   
253030  3275069               Kinetic           2022.0   2.0     57.0     NaN   
253031  3275113     Like Eating Glass              NaN   2.0      NaN     NaN   

        tag_id  tag_count  

In [ ]:


# ۱. لود کردن فایل‌های Last.fm با ستون‌های دقیق
# id در این فایل، همان شناسنامه خواننده در سایت Last.fm است
lastfm_artists = pd.read_csv(
    './data/artists.dat', 
    sep='\t', 
    usecols=['id', 'name']
)

# artistID در این فایل، دقیقاً به id بالا اشاره می‌کند
user_artists = pd.read_csv(
    './data/user_artists.dat', 
    sep='\t', 
    usecols=['artistID', 'weight']
)

# ۲. جمع زدن وزن‌ها (تعداد کل بازدیدها) برای هر artistID
# چون هر کاربر یک وزن داده، اول همه وزن‌های یک خواننده را با هم جمع می‌کنیم
weights_grouped = user_artists.groupby('artistID')['weight'].sum().reset_index()

# ۳. حالا وزن را به اسم خواننده می‌چسبانیم (در دنیای Last.fm)
# اینجا id از فایل اول را با artistID از فایل دوم ست می‌کنیم
lastfm_data = pd.merge(
    lastfm_artists, 
    weights_grouped, 
    left_on='id',      # از فایل artists.dat
    right_on='artistID', # از فایل user_artists.dat
    how='left'
).drop(columns=['artistID']) # ستون تکراری را حذف می‌کنیم

# اسم آیدی را برای وضوح بیشتر عوض می‌کنیم به lastfm_id
lastfm_data = lastfm_data.rename(columns={'id': 'lastfm_id', 'name': 'lastfm_name'})

# ۴. حالا این جدول (شامل اسم، آیدی لاست‌اف‌ام و وزن واقعی) را داریم.
# برای وصل کردن به جدول اصلی پروژه (final_artist_df) چون آیدی مشترک نداریم، 
# ناچاریم از اسم (Name) استفاده کنیم.

def normalize(s):
    if pd.isna(s): return ''
    return str(s).strip().lower()

final_artist_df['_name_norm'] = final_artist_df['name'].apply(normalize)
lastfm_data['_name_norm'] = lastfm_data['lastfm_name'].apply(normalize)

# ۵. انتقال آیدی و وزن به جدول اصلی
# اول ستون‌های قدیمی احتمالی را پاک می‌کنیم که MergeError ندهد
for col in ['lastfm_id', 'weight']:
    if col in final_artist_df.columns:
        final_artist_df = final_artist_df.drop(columns=[col])

final_artist_df = pd.merge(
    final_artist_df, 
    lastfm_data[['lastfm_id', 'weight', '_name_norm']], 
    on='_name_norm', 
    how='left'
)

# ۶. نهایی سازی
final_artist_df['weight'] = final_artist_df['weight'].fillna(0)
final_artist_df = final_artist_df.drop(columns=['_name_norm'])

print("✅ وزن‌ها بر اساس artistID صحیح جایگذاری شدند.")
# نمایش هنرمندانی که بیشترین بازدید (weight) را داشتند
final_artist_df.sort_values('weight', ascending=False).head(10)

✅ وزن‌ها بر اساس artistID صحیح جایگذاری شدند.


,id,name,begin_date_year,type,area,gender,tag_id,tag_count,rating,rating_count,lastfm_id,weight
335,791,Britney Spears,1981.0,1.0,222.0,2.0,111,7,86.0,18.0,289.0,2393140.0
161,317,Depeche Mode,1980.0,2.0,221.0,NaN,527,0,89.0,38.0,72.0,1301308.0
71789,531923,Lady Gaga,1986.0,1.0,222.0,2.0,251746,1,86.0,26.0,89.0,1291387.0
261,495,Christina Aguilera,1980.0,1.0,222.0,2.0,1060,3,93.0,8.0,292.0,1058405.0
41364,273945,Paramore,2004.0,2.0,222.0,NaN,8636,1,89.0,16.0,498.0,963449.0
53,89,Madonna,1958.0,1.0,222.0,2.0,39075,-2,88.0,42.0,67.0,921198.0
40127,262731,Rihanna,1988.0,1.0,222.0,2.0,1047,1,76.0,30.0,288.0,905423.0
573,1332,Shakira,1977.0,1.0,47.0,2.0,252431,0,97.0,13.0,701.0,688529.0
151,303,The Beatles,1960.0,2.0,221.0,NaN,1312,-4,96.0,85.0,227.0,662116.0
46222,315753,Katy Perry,1984.0,1.0,222.0,2.0,56565,-4,73.0,18.0,300.0,532545.0
